In [1]:
from pathlib import Path
from collections import defaultdict
import re

import pandas as pd
from ultralytics import YOLO

In [ ]:
model_path = "yolo_model_weights.pt"
tiles_folder = "tile_path"
output_excel = "tile_counts.xlsx"

count_classes = [1, 2]

In [3]:
model = YOLO(model_path)

tiles_path = Path(tiles_folder)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
image_files = sorted([p for p in tiles_path.iterdir() if p.suffix.lower() in image_extensions])

print(f"Found {len(image_files)} image tiles.")

Found 99 image tiles.


In [4]:
def extract_index(filename): # tile names are all formatted like index_{stuff}.png, this returns the index
    return int(filename.split("_")[0])

In [5]:
counts_by_index = defaultdict(lambda: defaultdict(int))

In [6]:
for img_path in image_files:
    tile_index = extract_index(img_path.name)

    results = model(str(img_path), verbose=False)

    for result in results:
        if result.boxes is None or result.boxes.cls is None:
            continue

        detected_classes = result.boxes.cls.cpu().numpy().astype(int)

        for cls_id in detected_classes:
            if cls_id in count_classes:
                counts_by_index[tile_index][cls_id] += 1

In [7]:
rows = []

for tile_index in sorted(counts_by_index.keys()):
    row = {"index": tile_index}
    for cls_id in count_classes:
        class_name = model.names[cls_id]
        row[class_name] = counts_by_index[tile_index].get(cls_id, 0)
    rows.append(row)

df = pd.DataFrame(rows)
df

,index,Atypical,Typical
0,101,0,7
1,102,2,20
2,103,2,12
3,104,6,23
4,105,18,18
...,...,...,...
93,195,0,1
94,196,39,19
95,197,1,4
96,198,14,13


In [8]:
df.to_excel(output_excel, index=False)
print(f"Saved spreadsheet to: {output_excel}")

Saved spreadsheet to: yolo_tile_counts.xlsx
